In [0]:
%sql
DROP TABLE IF EXISTS de_workspace26.bronze_aditya_sah_assessment.orders_stream;

In [0]:
dbutils.fs.rm(
    "/Volumes/de_workspace26/bronze_aditya_sah_assessment/raw_data/_checkpoints/orders_stream",
    True
)

dbutils.fs.rm(
    "/Volumes/de_workspace26/bronze_aditya_sah_assessment/raw_data/_schema/orders",
    True
)

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("region", StringType(), True)
])

In [0]:
from pyspark.sql.functions import current_timestamp, lit

base_path = "/Volumes/de_workspace26/bronze_aditya_sah_assessment/raw_data/"

orders_stream_df = spark.readStream.format("cloudFiles") \
    .schema(schema) \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option(
        "cloudFiles.schemaLocation",
        base_path + "_schema/orders"
    ) \
    .load(base_path + "orders/")


orders_stream_df = orders_stream_df.withColumn("_ingested_at", current_timestamp()) \
                                   .withColumn("_source_file", lit("orders_stream"))


query = (orders_stream_df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        base_path + "_checkpoints/orders_stream"
    ) \
    .outputMode("append") \
    .table("de_workspace26.bronze_aditya_sah_assessment.orders_stream"))

query.awaitTermination()
